In [22]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [21]:
df = pd.read_csv('loan.csv', low_memory=False)

print(f"Original dataset shape: {df.shape}")

Original dataset shape: (887379, 74)


In [23]:
# ---------------------------------------------------------
# PRO-TIP: Standardize column names to make coding easier
# ---------------------------------------------------------
# This makes all columns lowercase and replaces spaces with underscores
# so you don't get errors from weird formatting in your raw data.
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [24]:
# ==========================================
# STEP 1: Define Your Target Variable
# ==========================================
target_col = 'loan_status' 

if target_col in df.columns:
    # 1. Standardize the text in the status column to lowercase
    df[target_col] = df[target_col].astype(str).str.lower()
    
    # 2. Filter out active loans (handling various LendingClub text formats)
    active_statuses = ['current', 'in grace period', 'issued', 'late (16-30 days)', 'late (31-120 days)']
    df = df[~df[target_col].isin(active_statuses)]

    # 3. Create Binary Label (0 = Good, 1 = Bad)
    # Using a function to catch variations like "Does not meet the credit policy. Status:Charged Off"
    def map_status(status):
        if 'fully paid' in status:
            return 0
        elif 'charged off' in status or 'default' in status:
            return 1
        else:
            return np.nan # Flags anything we missed as missing data

    df[target_col] = df[target_col].apply(map_status)
    
    # Drop rows that didn't match our criteria
    df = df.dropna(subset=[target_col])
    
    print(f"\nShape after filtering target variable: {df.shape}")
    print("Target Variable Distribution (0=Good, 1=Bad):")
    print(df[target_col].value_counts())
else:
    print(f"\nERROR: Could not find '{target_col}' column. Please check your raw data headers.")


Shape after filtering target variable: (256939, 74)
Target Variable Distribution (0=Good, 1=Bad):
loan_status
0    209711
1     47228
Name: count, dtype: int64


In [25]:
# ==========================================
# STEP 2: Eliminate Data Leakage
# ==========================================
# A comprehensive list of ALL known LendingClub leakage columns
# (Features that are generated AFTER the loan is already issued)
leakage_columns = [
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 
    'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 
    'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d',
    'settlement_status', 'settlement_date', 'settlement_amount'
]

# This dynamically checks your dataset and only drops the ones that actually exist in your file
cols_to_drop = [col for col in leakage_columns if col in df.columns]
df = df.drop(columns=cols_to_drop)

print(f"\nSuccessfully dropped {len(cols_to_drop)} leakage columns.")
print(f"Shape after dropping leakage columns: {df.shape}")


Successfully dropped 13 leakage columns.
Shape after dropping leakage columns: (256939, 61)


In [26]:
# ==========================================
# STEP 3: Clean Clutter & Handle Missing Values
# ==========================================

# 1. Drop Useless Identifiers and Free-Text
# These columns are either unique to every user (like ID) or text that requires NLP to process.
useless_columns = ['id', 'member_id', 'url', 'desc', 'emp_title', 'title', 'zip_code']
cols_to_drop = [col for col in useless_columns if col in df.columns]
df = df.drop(columns=cols_to_drop)

# 2. Drop columns with too many missing values (Threshold: 50%)
# If a column is missing more than half its data, it's not reliable for ML.
threshold = len(df) * 0.5 
df = df.dropna(thresh=threshold, axis=1)

# 3. Impute (Fill) remaining missing values
# We fill numbers with the 'median' (less sensitive to outliers) 
# We fill categories/text with the 'mode' (most frequent value)
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object', 'string']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"Shape after Step 3 (Cleaning/Imputing): {df.shape}")
print(f"Total missing values remaining: {df.isnull().sum().sum()}")

Shape after Step 3 (Cleaning/Imputing): (256939, 34)
Total missing values remaining: 0


In [27]:
# ==========================================
# REVISED STEP 4: Encode Categorical Features
# ==========================================

print(f"Starting shape: {df.shape}")

# 1. DROP High-Cardinality (Too many unique values) and Date Columns
# This prevents our columns from exploding to 800+
cols_to_drop = ['issue_d', 'earliest_cr_line', 'addr_state', 'title', 'zip_code']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# 2. Clean and Encode 'emp_length'
if 'emp_length' in df.columns:
    df['emp_length'] = df['emp_length'].astype(str).str.extract(r'(\d+)').astype(float)
    df['emp_length'] = df['emp_length'].fillna(df['emp_length'].median())

# 3. Encode 'term' 
if 'term' in df.columns:
    df['term'] = df['term'].astype(str).str.extract(r'(\d+)').astype(float)

# 4. Ordinal Encoding for 'grade' 
if 'grade' in df.columns:
    grade_map = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7}
    df['grade'] = df['grade'].astype(str).str.lower().map(grade_map)
    df['grade'] = df['grade'].fillna(df['grade'].median())

if 'sub_grade' in df.columns:
    df = df.drop(columns=['sub_grade']) 

# 5. One-Hot Encoding for Nominal Categories
# Notice the addition of 'string' to suppress that warning you got!
remaining_cat_cols = df.select_dtypes(include=['object', 'string']).columns
print(f"\nApplying One-Hot Encoding to: {list(remaining_cat_cols)}")

df = pd.get_dummies(df, columns=remaining_cat_cols, drop_first=True)

print(f"\nFinal shape after Step 4 (Ready for ML prep): {df.shape}")

Starting shape: (256939, 34)

Applying One-Hot Encoding to: ['home_ownership', 'verification_status', 'pymnt_plan', 'purpose', 'initial_list_status', 'application_type']

Final shape after Step 4 (Ready for ML prep): (256939, 47)


In [28]:
# ==========================================
# STEP 5: Split, Scale, and Balance (SMOTE)
# ==========================================

print(f"Starting Step 5 with dataset shape: {df.shape}")

# 1. Separate Features (X) and Target (y)
# 'loan_status' is what we want to predict, everything else is a feature
X = df.drop(columns=['loan_status'])
y = df['loan_status']

# 2. Train-Test Split
# We keep 80% of data for training the model, and hide 20% for testing it later.
# stratify=y ensures the 80/20 split maintains the same ratio of good/bad loans.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining Data (Before SMOTE): {X_train.shape[0]} rows")
print(f"Testing Data: {X_test.shape[0]} rows")
print("Target distribution in training data (Before SMOTE):")
print(y_train.value_counts())

# 3. Feature Scaling (Standardization)
# This converts all columns to have a mean of 0 and standard deviation of 1.
scaler = StandardScaler()

# We 'fit' (learn the math) ONLY on the training data, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Do NOT fit on test data!

# Convert back to DataFrames so we don't lose our column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# 4. Handle Imbalanced Data using SMOTE
# We ONLY apply SMOTE to the training data. The test data must remain real!
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"\nTraining Data (After SMOTE): {X_train_balanced.shape[0]} rows")
print("Target distribution in training data (After SMOTE):")
print(y_train_balanced.value_counts())

print("\n--- PRE-PROCESSING COMPLETE! ---")
print("Your data is now 100% ready to be fed into your Machine Learning models.")

Starting Step 5 with dataset shape: (256939, 47)

Training Data (Before SMOTE): 205551 rows
Testing Data: 51388 rows
Target distribution in training data (Before SMOTE):
loan_status
0    167769
1     37782
Name: count, dtype: int64

Training Data (After SMOTE): 335538 rows
Target distribution in training data (After SMOTE):
loan_status
0    167769
1    167769
Name: count, dtype: int64

--- PRE-PROCESSING COMPLETE! ---
Your data is now 100% ready to be fed into your Machine Learning models.
